In [ ]:
# MiniProyecto 3: Predicción de ángulo de rotación en MNIST usando rotación en el
#           espacio latente (bloques 2D) + entrenamiento conjunto de pérdidas.
# Dadas dos imágenes (x0 y xt) donde xt es x0 rotada un ángulo t,
#           el modelo aprende a:
#             - comprimirlas a un espacio latente con un Encoder (E),
#             - predecir el ángulo t entre sus latentes (Tθ),
#             - rotar z0 en latente con ese ángulo,
#             - decodificar y reconstruir xt,
#             - y minimizar 4 pérdidas (recon/latente/rot_recon/ángulo).

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate

# 0) CONFIGURACIÓN Y UTILIDADES
# Fijamos semillas para reproducibilidad.

np.random.seed(42)
tf.random.set_seed(42)


# 1) GENERADOR DE DATOS: pares (x0, xt, t)
#    - Dado MNIST, forma tripletas:
#        x0: imagen original (28x28)
#        xt: x0 rotada en t grados
#        t : ángulo en radianes
#    - Usa padding antes de rotar para evitar recortes y aliasing.
#    - Devuelve tensores aplanados de 784 (=28*28) en [0,1].
#

class RotatedMNISTGenerator(keras.utils.Sequence):
    """
    Genera lotes ((x0, xt), t). x0 y xt son vectores de tamaño 784 en [0,1].
    t es el ángulo real de rotación en radianes.
    """
    def __init__(self, x_data, batch_size=128, angle_range=(-45, 45), shuffle=True):
        self.x_data = x_data                      # Imágenes MNIST (28x28)
        self.batch_size = batch_size              # Tamaño de lote
        self.angle_range = angle_range            # Rango de rotación en grados
        self.shuffle = shuffle                    # Si barajar por época
        self.indices = np.arange(len(x_data))     # Índices de imágenes
        self.on_epoch_end()

    def __len__(self):
        # Número de lotes por época
        return int(np.floor(len(self.x_data) / self.batch_size))

    def __getitem__(self, index):
        # Ensambla un lote a partir de índices
        batch_indices = self.indices[index * self.batch_size:(index + 1) * self.batch_size]
        return self._generate_batch(batch_indices)

    def on_epoch_end(self):
        # Barajado al final de cada época si corresponde
        if self.shuffle:
            np.random.shuffle(self.indices)

    def rotate_image(self, image, angle_degrees):
        """
        Rota una imagen 28x28:
          1) Hace padding a 36x36 (pad=4) para evitar recortes en bordes.
          2) Rota con interpolación cúbica (order=3) y fondo negro (cval=0).
          3) Recorta de vuelta al centro 28x28.
        """
        pad = 4
        padded = np.pad(image, ((pad, pad), (pad, pad)), mode='constant', constant_values=0)
        rotated = rotate(padded, angle_degrees, reshape=False, mode='constant', cval=0, order=3)
        start, end = pad, pad + 28
        return rotated[start:end, start:end]

    def _generate_batch(self, batch_indices):
        # Crea tensores del tamaño del lote
        batch_size = len(batch_indices)
        x0_batch = np.zeros((batch_size, 784), dtype=np.float32)
        xt_batch = np.zeros((batch_size, 784), dtype=np.float32)
        t_batch = np.zeros((batch_size,), dtype=np.float32)

        for i, idx in enumerate(batch_indices):
            img = self.x_data[idx].reshape(28, 28)                     # x0
            t_degrees = np.random.uniform(*self.angle_range)           # t en grados
            t_radians = np.deg2rad(t_degrees)                          # t en radianes
            img_rotated = self.rotate_image(img, t_degrees)            # xt

            x0_batch[i] = img.flatten()
            xt_batch[i] = img_rotated.flatten()
            t_batch[i] = t_radians

        # Normaliza a [0,1] (MNIST ya viene en [0,1] tras división, pero aseguramos)
        x0_batch = np.clip(x0_batch, 0, 1)
        xt_batch = np.clip(xt_batch, 0, 1)
        return (x0_batch, xt_batch), t_batch


# 2) SUBREDES: ENCODER, DECODER y PREDICTOR DE ÁNGULO
#    - Encoder: 784 -> ... -> latent_dim (comprimir imagen a "huella" latente)
#    - Decoder: latent_dim -> ... -> 784 (reconstruir imagen desde latente)
#    - Angle Predictor (Tθ): concatena z0 y zt y estima t_pred (radianes)

def build_encoder(latent_dim=32, name='encoder'):
    """MLP que comprime la imagen a un vector latente de tamaño latent_dim."""
    inputs = layers.Input(shape=(784,), name=f'{name}_input')
    x = layers.Dense(256, activation='relu', name=f'{name}_dense1')(inputs)
    x = layers.Dense(128, activation='relu', name=f'{name}_dense2')(x)
    x = layers.Dense(64, activation='relu', name=f'{name}_dense3')(x)
    z = layers.Dense(latent_dim, name=f'{name}_latent')(x)  # Latente sin activación
    return Model(inputs=inputs, outputs=z, name=name)

def build_decoder(latent_dim=32, name='decoder'):
    """MLP que reconstruye la imagen a partir del vector latente."""
    inputs = layers.Input(shape=(latent_dim,), name=f'{name}_input')
    x = layers.Dense(64, activation='relu', name=f'{name}_dense1')(inputs)
    x = layers.Dense(128, activation='relu', name=f'{name}_dense2')(x)
    x = layers.Dense(256, activation='relu', name=f'{name}_dense3')(x)
    outputs = layers.Dense(784, activation='sigmoid', name=f'{name}_output')(x)  # Pixeles [0,1]
    return Model(inputs=inputs, outputs=outputs, name=name)

def build_angle_predictor(latent_dim=32, name='angle_predictor'):
    """
    Red que recibe [z0, zt] (concatenados) y predice el ángulo t_pred (radianes).
    """
    z0_input = layers.Input(shape=(latent_dim,), name=f'{name}_z0')
    zt_input = layers.Input(shape=(latent_dim,), name=f'{name}_zt')
    z_concat = layers.Concatenate(name=f'{name}_concat')([z0_input, zt_input])
    x = layers.Dense(128, activation='relu', name=f'{name}_dense1')(z_concat)
    x = layers.Dense(64, activation='relu', name=f'{name}_dense2')(x)
    x = layers.Dense(32, activation='relu', name=f'{name}_dense3')(x)
    t_pred = layers.Dense(1, name=f'{name}_output')(x)  # Regresión en radianes
    return Model(inputs=[z0_input, zt_input], outputs=t_pred, name=name)


# 3) CAPA PERSONALIZADA: ROTACIÓN 2D EN LATENTE
#    - Interpreta el vector latente como pares 2D consecutivos: (z0,z1),(z2,z3)...
#    - Aplica la MISMA rotación 2D (cos/sin) a cada par con el ángulo t.
#    - Requiere que latent_dim sea par (assert en el modelo).

class RotateLatentLayer(layers.Layer):
    """Aplica rotación 2D a cada par (x,y) del vector latente z."""
    def __init__(self, **kwargs):
        super(RotateLatentLayer, self).__init__(**kwargs)

    def call(self, inputs):
        # inputs = [z, angle]; z: (batch, latent_dim), angle: (batch, 1)
        z, angle = inputs
        batch_size = tf.shape(z)[0]
        latent_dim = tf.shape(z)[1]
        angle = tf.squeeze(angle)  # (batch,)

        # Reorganiza z en pares 2D: (batch, latent_dim//2, 2)
        z_pairs = tf.reshape(z, [batch_size, latent_dim // 2, 2])

        # Matriz de rotación (cos/sin) broadcast para todos los pares
        cos_t = tf.reshape(tf.cos(angle), [-1, 1, 1])
        sin_t = tf.reshape(tf.sin(angle), [-1, 1, 1])

        x = z_pairs[:, :, 0:1]
        y = z_pairs[:, :, 1:2]
        x_rot = x * cos_t - y * sin_t
        y_rot = x * sin_t + y * cos_t

        # Reconstruye el vector latente rotado
        z_rotated = tf.concat([x_rot, y_rot], axis=2)
        z_rotated = tf.reshape(z_rotated, [batch_size, latent_dim])
        return z_rotated


# 4) MODELO COMPLETO + ENTRENAMIENTO PERSONALIZADO
#    - Integra E, D, Tθ y la capa de rotación.
#    - Define las 4 pérdidas + una regularización geométrica ligera.
#    - Expone métricas (incluye MAE de ángulo en grados para EarlyStopping).

class RotationLatentModel(Model):
    """Modelo global: E, D, Tθ y rotación en latente con entrenamiento custom."""
    def __init__(self, latent_dim=32, lambda1=1.0, lambda2=1.0, lambda3=0.1, **kwargs):
        super(RotationLatentModel, self).__init__(**kwargs)

        assert latent_dim % 2 == 0, "latent_dim debe ser par para rotar por pares 2D."

        # Pesos de pérdidas
        self.latent_dim = latent_dim
        self.lambda1 = lambda1  # Consistencia latente
        self.lambda2 = lambda2  # Reconstrucción del rotado
        self.lambda3 = lambda3  # Error de ángulo

        # Submodelos
        self.encoder = build_encoder(latent_dim)
        self.decoder = build_decoder(latent_dim)
        self.angle_predictor = build_angle_predictor(latent_dim)
        self.rotate_layer = RotateLatentLayer()

        # Métricas agregadas
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.recon_loss_tracker = keras.metrics.Mean(name="recon_loss")
        self.latent_loss_tracker = keras.metrics.Mean(name="latent_loss")
        self.rot_recon_loss_tracker = keras.metrics.Mean(name="rot_recon_loss")
        self.angle_loss_tracker = keras.metrics.Mean(name="angle_loss")
        self.angle_mae_tracker = keras.metrics.Mean(name="angle_mae_deg")  # en grados

    def call(self, inputs, training=False):
        """
        Forward pass:
          - E(x0)->z0, E(xt)->zt
          - Tθ([z0,zt])->t_pred
          - Rotación latente: z0 -> z0_rot usando t_pred
          - Decodificación: reconstrucciones x0_recon, xt_recon y xt_from_rot
        """
        x0, xt = inputs
        z0 = self.encoder(x0, training=training)
        zt = self.encoder(xt, training=training)
        t_pred = self.angle_predictor([z0, zt], training=training)
        z0_rot = self.rotate_layer([z0, t_pred])
        x0_recon = self.decoder(z0, training=training)
        xt_recon = self.decoder(zt, training=training)
        xt_from_rot = self.decoder(z0_rot, training=training)
        return {
            'z0': z0, 'zt': zt, 'z0_rot': z0_rot, 't_pred': t_pred,
            'x0_recon': x0_recon, 'xt_recon': xt_recon, 'xt_from_rot': xt_from_rot
        }

    def _pair_norm_regularizer(self, z0, z0_rot):
        """
        Regularización geométrica ligera:
          - Una rotación ideal preserva la norma de cada par 2D.
          - Penalizamos cambios de norma par-a-par (peso pequeño 1e-4).
        """
        z_pairs = tf.reshape(z0, [-1, self.latent_dim // 2, 2])
        z_rot_pairs = tf.reshape(z0_rot, [-1, self.latent_dim // 2, 2])
        norm_diff = tf.reduce_mean(tf.abs(tf.norm(z_pairs, axis=2) - tf.norm(z_rot_pairs, axis=2)))
        return 1e-4 * norm_diff

    def train_step(self, data):
        """
        Entrenamiento:
          L_total = L_recon + λ1*L_latente + λ2*L_rot_recon + λ3*L_ángulo + reg_par
        """
        (x0, xt), t_real = data
        with tf.GradientTape() as tape:
            outputs = self([x0, xt], training=True)

            # 1) Reconstrucción: AE sobre x0 y xt
            recon_loss = (
                tf.reduce_mean(tf.square(x0 - outputs['x0_recon'])) +
                tf.reduce_mean(tf.square(xt - outputs['xt_recon']))
            )
            # 2) Consistencia latente: z0_rot ≈ zt
            latent_loss = tf.reduce_mean(tf.square(outputs['z0_rot'] - outputs['zt']))
            # 3) Reconstrucción del rotado: D(z0_rot) ≈ xt
            rot_recon_loss = tf.reduce_mean(tf.square(xt - outputs['xt_from_rot']))
            # 4) Error angular: |t_pred - t_real|
            t_pred_squeezed = tf.squeeze(outputs['t_pred'])
            angle_loss = tf.reduce_mean(tf.abs(t_pred_squeezed - t_real))
            # (+) Regularización par-a-par (norma)
            pair_norm_reg = self._pair_norm_regularizer(outputs['z0'], outputs['z0_rot'])

            total_loss = (
                recon_loss +
                self.lambda1 * latent_loss +
                self.lambda2 * rot_recon_loss +
                self.lambda3 * angle_loss +
                pair_norm_reg
            )

        grads = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))

        # Actualiza métricas visibles durante el fit()
        self.loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.latent_loss_tracker.update_state(latent_loss)
        self.rot_recon_loss_tracker.update_state(rot_recon_loss)
        self.angle_loss_tracker.update_state(angle_loss)
        angle_mae_deg = tf.reduce_mean(tf.abs(t_pred_squeezed - t_real)) * 180.0 / np.pi
        self.angle_mae_tracker.update_state(angle_mae_deg)

        return {
            "loss": self.loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "latent_loss": self.latent_loss_tracker.result(),
            "rot_recon_loss": self.rot_recon_loss_tracker.result(),
            "angle_loss": self.angle_loss_tracker.result(),
            "angle_mae_deg": self.angle_mae_tracker.result(),
        }

    def test_step(self, data):
        """
        Evaluación (sin actualizar pesos) calculando exactamente las mismas pérdidas.
        """
        (x0, xt), t_real = data
        outputs = self([x0, xt], training=False)

        recon_loss = (
            tf.reduce_mean(tf.square(x0 - outputs['x0_recon'])) +
            tf.reduce_mean(tf.square(xt - outputs['xt_recon']))
        )
        latent_loss = tf.reduce_mean(tf.square(outputs['z0_rot'] - outputs['zt']))
        rot_recon_loss = tf.reduce_mean(tf.square(xt - outputs['xt_from_rot']))
        t_pred_squeezed = tf.squeeze(outputs['t_pred'])
        angle_loss = tf.reduce_mean(tf.abs(t_pred_squeezed - t_real))
        pair_norm_reg = self._pair_norm_regularizer(outputs['z0'], outputs['z0_rot'])

        total_loss = (
            recon_loss +
            self.lambda1 * latent_loss +
            self.lambda2 * rot_recon_loss +
            self.lambda3 * angle_loss +
            pair_norm_reg
        )

        self.loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.latent_loss_tracker.update_state(latent_loss)
        self.rot_recon_loss_tracker.update_state(rot_recon_loss)
        self.angle_loss_tracker.update_state(angle_loss)
        angle_mae_deg = tf.reduce_mean(tf.abs(t_pred_squeezed - t_real)) * 180.0 / np.pi
        self.angle_mae_tracker.update_state(angle_mae_deg)

        return {
            "loss": self.loss_tracker.result(),
            "recon_loss": self.recon_loss_tracker.result(),
            "latent_loss": self.latent_loss_tracker.result(),
            "rot_recon_loss": self.rot_recon_loss_tracker.result(),
            "angle_loss": self.angle_loss_tracker.result(),
            "angle_mae_deg": self.angle_mae_tracker.result(),
        }

    @property
    def metrics(self):
        # Lista de métricas que Keras resetea/gestiona por época
        return [
            self.loss_tracker,
            self.recon_loss_tracker,
            self.latent_loss_tracker,
            self.rot_recon_loss_tracker,
            self.angle_loss_tracker,
            self.angle_mae_tracker,
        ]


# 5) EVALUACIÓN Y VISUALIZACIÓN
#    - evaluate_model: calcula MAE global y devuelve arrays de t_real/t_pred.
#    - visualize_results: muestra tripletas (x0, xt, D(z0_rot)) + ángulos.
#    - plot_angle_scatter: dispersión t_real vs t_pred y línea ideal.
#    - plot_training_history: pérdida total y MAE angular por época.

def evaluate_model(model, generator):
    t_real_list, t_pred_list = [], []
    for i in range(len(generator)):
        (x0, xt), t_real = generator[i]
        outputs = model([x0, xt], training=False)
        t_pred = outputs['t_pred'].numpy().squeeze()
        t_real_list.append(t_real)
        t_pred_list.append(t_pred)
    t_real_all = np.concatenate(t_real_list)
    t_pred_all = np.concatenate(t_pred_list)
    mae_radians = np.mean(np.abs(t_real_all - t_pred_all))
    mae_degrees = np.rad2deg(mae_radians)
    return {'mae_degrees': mae_degrees, 'mae_radians': mae_radians,
            't_real': t_real_all, 't_pred': t_pred_all}

def visualize_results(model, generator, num_samples=5):
    (x0, xt), t_real = generator[0]
    x0, xt, t_real = x0[:num_samples], xt[:num_samples], t_real[:num_samples]
    outputs = model([x0, xt], training=False)
    xt_from_rot = outputs['xt_from_rot'].numpy()[:num_samples]
    t_pred = outputs['t_pred'].numpy().squeeze()[:num_samples]

    x0 = x0.reshape(num_samples, 28, 28)
    xt = xt.reshape(num_samples, 28, 28)
    xt_from_rot = xt_from_rot.reshape(num_samples, 28, 28)

    fig, axes = plt.subplots(num_samples, 3, figsize=(9, 3*num_samples))
    for i in range(num_samples):
        axes[i, 0].imshow(x0[i], cmap='gray'); axes[i, 0].set_title('Original x0'); axes[i, 0].axis('off')
        axes[i, 1].imshow(xt[i], cmap='gray'); axes[i, 1].set_title(f'Rotado real xt\nt={np.rad2deg(t_real[i]):.1f}°'); axes[i, 1].axis('off')
        axes[i, 2].imshow(xt_from_rot[i], cmap='gray'); axes[i, 2].set_title(f'D(z0_rot)\nt_pred={np.rad2deg(t_pred[i]):.1f}°'); axes[i, 2].axis('off')
    plt.tight_layout(); plt.show()

def plot_angle_scatter(results):
    t_real_deg = np.rad2deg(results['t_real'])
    t_pred_deg = np.rad2deg(results['t_pred'])
    plt.figure(figsize=(8, 8))
    plt.scatter(t_real_deg, t_pred_deg, alpha=0.5, s=10)
    plt.plot([-45, 45], [-45, 45], '--', label='Predicción perfecta')
    plt.xlabel('Ángulo real (grados)'); plt.ylabel('Ángulo predicho (grados)')
    plt.title(f'Predicción de ángulos\nMAE = {results["mae_degrees"]:.2f}°')
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.axis('equal'); plt.xlim(-50, 50); plt.ylim(-50, 50)
    plt.show()

def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].plot(history.history['loss'], label='Train Loss')
    if 'val_loss' in history.history:
        axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Pérdida Total')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(history.history['angle_mae_deg'], label='Train Angle MAE')
    if 'val_angle_mae_deg' in history.history:
        axes[1].plot(history.history['val_angle_mae_deg'], label='Val Angle MAE')
    axes[1].axhline(y=10, linestyle='--', label='Meta: 10°')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Angle MAE (degrees)')
    axes[1].set_title('Error Absoluto Medio del Ángulo')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()


# 6) SCRIPT PRINCIPAL
#    - Carga MNIST, crea generadores, preentrena AE, entrena modelo conjunto,
#      evalúa y grafica, y guarda pesos.

def main():
    # Hiperparámetros principales
    LATENT_DIM = 32           # Debe ser par (pares 2D)
    BATCH_SIZE = 128
    EPOCHS_PREAE = 5          # Pre-entrenamiento del autoencoder (recomendado)
    EPOCHS = 20               # Épocas del modelo conjunto
    LEARNING_RATE = 1e-3
    LAMBDA1, LAMBDA2, LAMBDA3 = 1.0, 1.0, 0.1   # Pesos pérdidas (ajustables)

    print("TensorFlow:", tf.__version__)
    print("GPUs:", tf.config.list_physical_devices('GPU'))

    # 1) Cargar MNIST y normalizar a [0,1]
    (x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
    x_train = x_train.astype('float32') / 255.0
    x_test  = x_test.astype('float32') / 255.0
    print(f"\nDatos: Train {x_train.shape}, Test {x_test.shape}")

    # 2) Generadores de pares rotados (train y validación)
    train_generator = RotatedMNISTGenerator(x_train, batch_size=BATCH_SIZE, angle_range=(-45, 45), shuffle=True)
    test_generator  = RotatedMNISTGenerator(x_test,  batch_size=BATCH_SIZE, angle_range=(-45, 45), shuffle=False)
    print(f"\nBatches: Train={len(train_generator)} | Test={len(test_generator)}")

    # 3) Instanciar el modelo completo (E, D, Tθ, rotación latente)
    model = RotationLatentModel(latent_dim=LATENT_DIM, lambda1=LAMBDA1, lambda2=LAMBDA2, lambda3=LAMBDA3)

    # === Pre-entrenamiento del Autoencoder (opcional pero útil para estabilidad) ===
    ae_in = keras.Input(shape=(784,))
    z = model.encoder(ae_in)
    ae_out = model.decoder(z)
    ae = keras.Model(ae_in, ae_out, name="autoencoder")
    ae.compile(optimizer=keras.optimizers.Adam(LEARNING_RATE), loss="mse")

    x_train_flat = x_train.reshape(-1, 784)
    x_test_flat  = x_test.reshape(-1, 784)
    print("\n=== Pre-entrenando Autoencoder (calentamiento de E/D) ===")
    ae.fit(x_train_flat, x_train_flat, validation_data=(x_test_flat, x_test_flat),
           epochs=EPOCHS_PREAE, batch_size=256, verbose=1)

    # 4) Compilar el modelo conjunto (clipnorm para estabilidad de gradiente)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, clipnorm=1.0))

    # Mostrar arquitectura
    print("\n=== Arquitectura del modelo ===")
    print("\nEncoder:"); model.encoder.summary()
    print("\nDecoder:"); model.decoder.summary()
    print("\nAngle Predictor:"); model.angle_predictor.summary()

    # 5) Callbacks: parada temprana por MAE angular y reducción de LR en plateaus
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_angle_mae_deg', patience=5,
                                      restore_best_weights=True, verbose=1, mode='min'),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=3, min_lr=1e-6, verbose=1)
    ]

    # 6) Entrenamiento conjunto (rotación en latente)
    print("\n=== Entrenando modelo conjunto ===\n")
    history = model.fit(train_generator, epochs=EPOCHS,
                        validation_data=test_generator, callbacks=callbacks, verbose=1)

    # 7) Evaluación final en test
    print("\n=== Evaluación final ===")
    results = evaluate_model(model, test_generator)
    print(f"MAE final en test: {results['mae_degrees']:.2f}°")
    if results['mae_degrees'] < 10:
        print("✓ Meta alcanzada: MAE < 10°")
    else:
        print("✗ Meta no alcanzada aún. Ajusta lambdas o entrena más épocas.")

    # 8) Visualizaciones (curvas, tripletas y dispersión)
    print("\nGenerando visualizaciones...")
    plot_training_history(history)
    visualize_results(model, test_generator, num_samples=5)
    plot_angle_scatter(results)

    # 9) Guardar pesos
    model.save_weights('rotation_latent_model_tf.weights.h5')
    print("\nPesos guardados en 'rotation_latent_model_tf.weights.h5'")

    return model, history, results


# Punto de entrada del script
if __name__ == "__main__":
    model, history, results = main()

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

Datos: Train (60000, 28, 28), Test (10000, 28, 28)

Batches: Train=468 | Test=78

=== Pre-entrenando Autoencoder (calentamiento de E/D) ===
Epoch 1/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - loss: 0.0592 - val_loss: 0.0337
Epoch 2/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0272 - val_loss: 0.0228
Epoch 3/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0208 - val_loss: 0.0188
Epoch 4/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.0180 - val_loss: 0.0169
Epoch 5/5
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.0164 - val_loss: 0.0156

=== Arquitectura del modelo ===

Encoder:


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_input (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense1 (Dense)          │ (None, 256)            │       200,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense2 (Dense)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_dense3 (Dense)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_latent (Dense)          │ (None, 32)             │         2,080 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 244,192 (953.88 KB)

 Trainable params: 244,192 (953.88 KB)

...

 Trainable params: 18,689 (73.00 KB)

 Non-trainable params: 0 (0.00 B)


=== Entrenando modelo conjunto ===

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


468/468 ━━━━━━━━━━━━━━━━━━━━ 40s 67ms/step - angle_loss: 1.1496 - angle_mae_deg: 65.8683 - latent_loss: 1.6171 - loss: 1.8735 - recon_loss: 0.0912 - rot_recon_loss: 0.0502 - val_angle_loss: 0.0755 - val_angle_mae_deg: 4.3286 - val_latent_loss: 0.0036 - val_loss: 0.1179 - val_recon_loss: 0.0691 - val_rot_recon_loss: 0.0376 - learning_rate: 0.0010
Epoch 2/20
242/468 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - angle_loss: 0.0713 - angle_mae_deg: 4.0878 - latent_loss: 0.0036 - loss: 0.1142 - recon_loss: 0.0672 - rot_recon_loss: 0.0363